# Credit Scoring with PyTorch + Feast Feature Store

This notebook demonstrates how to replace the sklearn DecisionTree in the credit scoring tutorial with a **PyTorch neural network** while keeping Feast as the single source of truth for features.

## Architecture
```
Offline Store (Parquet) ──► get_historical_features ──► PyTorch Training
Online Store  (Redis)   ──► get_online_features     ──► PyTorch Inference
```

## Prerequisites
- Feast feature store initialized (`feast apply` already run)
- Redis running on localhost:6379
- Features materialized (`feast materialize-incremental`)

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch feast scikit-learn pandas joblib

## Cell 2: Initialize Feast and Load Raw Loan Data

In [ ]:
import pandas as pd
import numpy as np
import torch
from feast import FeatureStore

fs = FeatureStore(repo_path="../feature_repo")

# Load raw loan application data (labels + entity keys)
loans = pd.read_parquet("../feature_repo/data/loan_table.parquet")

print(f"Loan records: {len(loans)}")
print(f"Columns: {loans.columns.tolist()}")
print(f"Fraud rate: {loans['loan_status'].mean():.1%}")
loans.head()

## Cell 3: Retrieve Point-in-Time Correct Training Features from Feast

In [ ]:
feast_features = [
    "zipcode_features:city",
    "zipcode_features:state",
    "zipcode_features:location_type",
    "zipcode_features:tax_returns_filed",
    "zipcode_features:population",
    "zipcode_features:total_wages",
    "credit_history:credit_card_due",
    "credit_history:mortgage_due",
    "credit_history:student_loan_due",
    "credit_history:vehicle_loan_due",
    "credit_history:hard_pulls",
    "credit_history:missed_payments_2y",
    "credit_history:missed_payments_1y",
    "credit_history:missed_payments_6m",
    "credit_history:bankruptcies",
    "total_debt_calc:total_debt_due",
]

# Point-in-time correct join — no data leakage
training_df = fs.get_historical_features(
    entity_df=loans,
    features=feast_features,
).to_df()

print(f"Training dataframe shape: {training_df.shape}")
training_df.head()

## Cell 4: Preprocess Features

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

categorical_features = ["person_home_ownership", "loan_intent", "city", "state", "location_type"]
target = "loan_status"

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
scaler = StandardScaler()

df = training_df.copy().dropna()
encoder.fit(df[categorical_features])
df[categorical_features] = encoder.transform(df[categorical_features])

drop_cols = [target, "event_timestamp", "created_timestamp", "loan_id", "zipcode", "dob_ssn"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.reindex(sorted(X.columns), axis=1).fillna(0)
y = df[target]

X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Input features: {X.shape[1]}")

## Cell 5: Define and Train PyTorch Neural Network

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

class CreditNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

input_dim = X_train.shape[1]
model = CreditNet(input_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCELoss()

X_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_t = torch.tensor(y_train.values, dtype=torch.float32).to(device)

model.train()
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(X_t)
    loss = criterion(preds, y_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50  loss={loss.item():.4f}")

print("\n✅ Training complete")

## Cell 6: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    probs = model(X_test_t).cpu().numpy()

preds = (probs >= 0.5).astype(int)
auc = roc_auc_score(y_test, probs)

print(f"AUC-ROC: {auc:.4f}")
print()
print(classification_report(y_test, preds, target_names=["Rejected", "Approved"]))

## Cell 7: Materialize Features to Online Store (Redis)

In [ ]:
from datetime import datetime, timezone

fs.materialize_incremental(end_date=datetime.now(tz=timezone.utc))
print("✅ Features materialized to Redis online store")

## Cell 8: Real-Time Inference via Feast Online Store

In [ ]:
import joblib

# Simulate a new loan application arriving in real time
new_application = {
    "zipcode": [76104],
    "dob_ssn": ["19630621_4278"],
    "loan_amnt": [15000],
    "person_home_ownership": ["RENT"],
    "loan_intent": ["PERSONAL"],
}

# Retrieve features from Feast online store (low-latency Redis)
online_features = fs.get_online_features(
    entity_rows=[{
        "zipcode": new_application["zipcode"][0],
        "dob_ssn": new_application["dob_ssn"][0],
        "loan_amnt": new_application["loan_amnt"][0],
    }],
    features=feast_features,
).to_dict()

print("Online features from Feast:")
for k, v in online_features.items():
    print(f"  {k}: {v[0]}")

In [ ]:
# Build feature vector and run PyTorch inference
features = new_application.copy()
features.update(online_features)
df_infer = pd.DataFrame.from_dict(features)
df_infer[categorical_features] = encoder.transform(df_infer[categorical_features])
df_infer = df_infer.drop(columns=["zipcode", "dob_ssn"], errors="ignore")
df_infer = df_infer.reindex(sorted(df_infer.columns), axis=1).fillna(0)
X_infer = scaler.transform(df_infer)

model.eval()
with torch.no_grad():
    prob = model(torch.tensor(X_infer, dtype=torch.float32).to(device)).item()

decision = "✅ APPROVED" if prob >= 0.5 else "❌ REJECTED"
print(f"\nLoan Decision: {decision}")
print(f"Approval Probability: {prob:.4f}")

## Cell 9: Summary

| Step | Tool |
|---|---|
| Feature definitions | Feast `FeatureView`, `OnDemandFeatureView` |
| Training data | `get_historical_features` (point-in-time correct) |
| Online serving | `get_online_features` (Redis, low-latency) |
| Model | PyTorch `CreditNet` (64→32→1 with BatchNorm + Dropout) |
| No skew | Same Feast features used in both training and inference |